# 一、非同构数据合并
### 1.1 理解left、right、inner、outer
![](on.jpg)

# 可以这样形象的去理解：
## （1）inner join = **两张名单都出现的人**

想象一下社区搞活动，你有：

* **报名名单（左表）**
* **现场签到名单（右表）**

inner 就是：

> **只有既报了名，又来现场的人会被统计进入正式名单。**

也就是“两边都出现”。

和图里的**重叠区域**完全一致。

### 什么时候用？

* 对账
* 数据严格匹配
* 留下“双重确认”的数据

---

# （2）left join = **以“报名名单”为主，补充签到信息**

left join 是：

> **报名名单里的人一个都不能漏，
> 能找到签到信息就补上，找不到就留空。**

典型场景：

* 以左表为核心人群（如客户池、员工名册）
* 右表只是附加信息
* 匹配不上就留 NaN（表示缺失或未发生）

非常自然。

---

# （3）right join = **以“签到名单”为主**

right join 就是 left 的镜像：

> **现场签到的所有人都算，
> 报名名单对得上就补上，对不上就留空。**

使用场景少，但逻辑一样。

---

# （4）outer join = **两份名单全部纳入，不漏任何人**

outer 就是：

> **所有报过名的 + 所有来过现场的，
> 无论是否重叠，都全部保留。**

典型场景：

* 数据清洗：要得到"所有可能的人/事件集合"
* 需要完整覆盖，不漏任何一方

---

# （5）concat = **把两个名单直接上下拼接**

concat 就不是匹配，而是：

> **把两份格式一样的名单直接叠在一起，就像把第二页放在第一页下面。**

适用于：

* 两个月的同格式销量
* 多部门汇总 list
* 多文件批量加载

### **题 1：inner join - 找出既报名又到场的人**

```python
df_signup = pd.DataFrame({
    'name': ['Tom', 'Alice', 'Bob'],
    'group': ['A', 'A', 'B']
})

df_checkin = pd.DataFrame({
    'name': ['Alice', 'Bob', 'David'],
    'status': ['ok', 'ok', 'late']
})
```

要求：
使用 inner join 找出 **既报名又到场** 的人。

---

### **题 2：left join - 以报名名单为主**

用 left join 补充签到信息，出现 NaN 的原因是什么？

---

### **题 3：outer join - 社区大活动所有人记录**

生成包含所有出现过的人（报名 + 到场）的名单。

---

### **题 4：concat - 合并两天活动的签到表**

```python
day1 = pd.DataFrame({'name': ['Tom', 'Alice'], 'day': [1, 1]})
day2 = pd.DataFrame({'name': ['Bob'], 'day': [2]})
```

要求：
将 day1、day2 纵向拼接在一起。

---

### **题 5（进阶）：多键合并**

社区两个活动都有分组：

```python
dfA = pd.DataFrame({
    'name': ['Tom', 'Tom', 'Alice'],
    'day': [1, 2, 1],
    'score': [80, 82, 90]
})

dfB = pd.DataFrame({
    'name': ['Tom', 'Alice', 'Alice'],
    'day': [1, 1, 2],
    'reward': [5, 8, 10]
})
```

要求：
以 `name + day` 为键进行 merge。

---

### 1.2 pd.merge(left, right, how, on)
- left, right 待合并的df
- on 根据哪个字段进行合并
- how 合并方式,默认使用的inner。how包括left、right、inner、outer
![](img/merge.png)


In [1]:
import pandas as pd


adf = pd.DataFrame(
    {'x1': ['A', 'B', 'C'],
     'x2': [1, 2, 3]}
                  )

adf

,x1,x2
0,A,1
1,B,2
2,C,3


In [2]:
bdf = pd.DataFrame(
    {'x1': ['A', 'B', 'D'],
     'x3': ['T', 'F', 'T']}
                  )

bdf

,x1,x3
0,A,T
1,B,F
2,D,T


In [3]:
import pandas as pd

df1 = pd.merge(adf, bdf, how='left', on='x1')
df1

,x1,x2,x3
0,A,1,T
1,B,2,F
2,C,3,NaN


In [4]:
df2 = pd.merge(adf, bdf, how='right', on='x1')
df2

,x1,x2,x3
0,A,1.0,T
1,B,2.0,F
2,D,NaN,T


In [5]:
df3 = pd.merge(adf, bdf, how='inner', on='x1')
df3

,x1,x2,x3
0,A,1,T
1,B,2,F


In [6]:
df4 = pd.merge(adf, bdf, how='outer', on='x1')
df4

,x1,x2,x3
0,A,1.0,T
1,B,2.0,F
2,C,3.0,NaN
3,D,NaN,T


In [7]:
df = pd.merge(adf, bdf, on='x1')
df

,x1,x2,x3
0,A,1,T
1,B,2,F


# 二、同构数据合并
字段相同的dataframe合并

pd.concat([df1,df2])  将df1和df2合并

In [7]:
import pandas as pd

df1 = pd.read_csv('reshape/auto-mpg1.csv')
df1

,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,car name
0,15.0,6,258,110,3730,19.0,75,1,amc matador
1,18.0,6,225,95,3785,19.0,75,1,plymouth fury
2,21.0,6,231,110,3039,15.0,75,1,buick skyhawk
3,20.0,8,262,110,3221,13.5,75,1,chevrolet monza 2+2
4,13.0,8,302,129,3169,12.0,75,1,ford mustang ii
...,...,...,...,...,...,...,...,...,...
231,27.0,4,140,86,2790,15.6,82,1,ford mustang gl
232,44.0,4,97,52,2130,24.6,82,2,vw pickup
233,32.0,4,135,84,2295,11.6,82,1,dodge rampage
234,28.0,4,120,79,2625,18.6,82,1,ford ranger


In [8]:
df2 = pd.read_csv('reshape/auto-mpg2.csv')
df2

,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,car name
0,18.0,8,307.0,130,3504,12.0,70,1,chevrolet chevelle malibu
1,15.0,8,350.0,165,3693,11.5,70,1,buick skylark 320
2,18.0,8,318.0,150,3436,11.0,70,1,plymouth satellite
3,16.0,8,304.0,150,3433,12.0,70,1,amc rebel sst
4,17.0,8,302.0,140,3449,10.5,70,1,ford torino
...,...,...,...,...,...,...,...,...,...
191,22.0,6,225.0,100,3233,15.4,76,1,plymouth valiant
192,22.0,6,250.0,105,3353,14.5,76,1,chevrolet nova
193,24.0,6,200.0,81,3012,17.6,76,1,ford maverick
194,22.5,6,232.0,90,3085,17.6,76,1,amc hornet


In [9]:
df3 = pd.concat([df1, df2])
df3

,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,car name
0,15.0,6,258.0,110,3730,19.0,75,1,amc matador
1,18.0,6,225.0,95,3785,19.0,75,1,plymouth fury
2,21.0,6,231.0,110,3039,15.0,75,1,buick skyhawk
3,20.0,8,262.0,110,3221,13.5,75,1,chevrolet monza 2+2
4,13.0,8,302.0,129,3169,12.0,75,1,ford mustang ii
...,...,...,...,...,...,...,...,...,...
191,22.0,6,225.0,100,3233,15.4,76,1,plymouth valiant
192,22.0,6,250.0,105,3353,14.5,76,1,chevrolet nova
193,24.0,6,200.0,81,3012,17.6,76,1,ford maverick
194,22.5,6,232.0,90,3085,17.6,76,1,amc hornet
